# GEC inline-edit fine-tuning — SFT (Qwen2.5-3B-Instruct + Unsloth)

**Inputs:** `data/processed/train.jsonl` produced by `scripts/build_dataset.py`.

**Output:** a LoRA adapter pushed to `<your-hf-username>/qwen2.5-3b-gec-sft`.

Runs on a free Colab T4 (16 GB). Wall time ≈ 45–60 min for 10 k examples × 2 epochs.

## 1. Install dependencies

In [ ]:
%%capture
!pip install -q unsloth
# Unsloth pins compatible torch / xformers / trl / peft — let it manage them.
!pip install -q --no-deps trl peft accelerate bitsandbytes
!pip install -q datasets huggingface_hub

## 2. Fetch the training data
Clone the project repo so we can read `data/processed/train.jsonl`. Replace the URL with your fork if you renamed it.

In [ ]:
import os
REPO_URL = os.environ.get('GEC_REPO_URL', 'https://github.com/LittleHydron/gec-inline')
!test -d gec-inline || git clone --depth 1 $REPO_URL
%cd gec-inline
!ls data/processed/

## 3. Load Qwen2.5-3B-Instruct in 4-bit with a LoRA head

In [ ]:
from unsloth import FastLanguageModel

MAX_SEQ_LEN = 1024
BASE_MODEL = 'unsloth/Qwen2.5-3B-Instruct-bnb-4bit'

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = BASE_MODEL,
    max_seq_length = MAX_SEQ_LEN,
    load_in_4bit = True,
)
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    lora_alpha = 32,
    lora_dropout = 0.0,
    bias = 'none',
    use_gradient_checkpointing = 'unsloth',
    random_state = 3407,
)

## 4. Format the data with the Qwen chat template
Each row's `messages` field already has the system + user + assistant turns. We render them through `tokenizer.apply_chat_template` and let Unsloth's `train_on_responses_only` mask the prompt out of the loss.

In [ ]:
from datasets import load_dataset

raw = load_dataset('json', data_files='data/processed/train.jsonl', split='train')
print('rows:', len(raw))
print('example messages:', raw[0]['messages'])

def format_example(ex):
    text = tokenizer.apply_chat_template(ex['messages'], tokenize=False)
    return {'text': text}

ds = raw.map(format_example, remove_columns=raw.column_names)
print(ds[0]['text'][:600])

## 5. Configure the SFT trainer

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth import is_bfloat16_supported
from unsloth.chat_templates import train_on_responses_only

# T4 is Turing -> fp16 only. A100/L4/H100 are Ampere+ -> bf16.
USE_BF16 = is_bfloat16_supported()

config = SFTConfig(
    output_dir = 'outputs/sft',
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 4,
    warmup_ratio = 0.03,
    num_train_epochs = 2,
    learning_rate = 2e-4,
    lr_scheduler_type = 'cosine',
    weight_decay = 0.01,
    optim = 'adamw_8bit',
    logging_steps = 20,
    save_strategy = 'epoch',
    save_total_limit = 1,
    seed = 3407,
    bf16 = USE_BF16,
    fp16 = not USE_BF16,
    max_seq_length = MAX_SEQ_LEN,
    dataset_text_field = 'text',
    packing = False,
    report_to = 'none',
)

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = ds,
    args = config,
)

# Mask the prompt out of the loss so only the assistant span is trained.
# (Tags below are Qwen2.5's chat-template instruction markers.)
trainer = train_on_responses_only(
    trainer,
    instruction_part = '<|im_start|>user\n',
    response_part = '<|im_start|>assistant\n',
)

## 6. Train

In [ ]:
stats = trainer.train()
stats.metrics

## 7. Sanity check the model
Run a couple of sentences through the trained model and confirm we see the bracketed format.

In [ ]:
from unsloth import FastLanguageModel
FastLanguageModel.for_inference(model)

from gec.prompts import build_chat_messages

for s in ['I goes to school every day .',
          'She have did her homework already .',
          'The cats was sleeping on the rug .']:
    prompt = tokenizer.apply_chat_template(
        build_chat_messages(s),
        tokenize=False, add_generation_prompt=True,
    )
    inputs = tokenizer(prompt, return_tensors='pt').to('cuda')
    out = model.generate(**inputs, max_new_tokens=128, do_sample=False)
    print(tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True))
    print('---')

## 8. Push the adapter to the HuggingFace Hub
Generate an HF access token at https://huggingface.co/settings/tokens (write scope) and paste it when prompted.

In [ ]:
from huggingface_hub import login
login()

HF_USER = 'Lopato4ka'  # <- EDIT if you are not Lopato4ka
ADAPTER_REPO = f'{HF_USER}/qwen2.5-3b-gec-sft'
MERGED_REPO  = f'{HF_USER}/qwen2.5-3b-gec-sft-merged'

model.push_to_hub(ADAPTER_REPO, private=False)
tokenizer.push_to_hub(ADAPTER_REPO, private=False)
# Merged 16-bit copy: the DPO stage loads THIS as its base so that
# 'adapter disabled' == the SFT policy (the frozen DPO reference).
model.push_to_hub_merged(MERGED_REPO, tokenizer, save_method='merged_16bit', private=False)
print('pushed:', ADAPTER_REPO, 'and', MERGED_REPO)

## 9. (Optional) Generate predictions on the eval set
Doing this here saves you re-loading the model later for `scripts/eval.py`.

In [ ]:
import json, sys
from pathlib import Path
from tqdm import tqdm
from gec.inference import generate_batch

EVAL_PATH = 'data/processed/eval_bea_dev.jsonl'
OUT_PATH  = 'results/predictions/sft_bea_dev.jsonl'
Path(OUT_PATH).parent.mkdir(parents=True, exist_ok=True)

rows = [json.loads(line) for line in open(EVAL_PATH)]
sentences = [r['source'] for r in rows]

results = []
for start in tqdm(range(0, len(sentences), 8)):
    batch = sentences[start:start+8]
    out = generate_batch(batch, tokenizer, model, batch_size=8)
    for r in out:
        results.append({'source': r.source, 'raw': r.raw,
                        'corrected': r.corrected, 'parse_ok': r.parse_ok})

with open(OUT_PATH, 'w') as f:
    for r in results:
        f.write(json.dumps(r) + '\n')
print('wrote', OUT_PATH, len(results))